# Pipeline 2: Donor Churn Prevention

## 1. Problem Framing

**Business Question:** How do we prevent donor churn year after year?

**Stakeholder:** Fundraising leadership who need to identify at-risk donors before they lapse, so the team can intervene with targeted re-engagement.

**Why it matters:** With only ~60 supporters, losing even a few donors can significantly impact the organization's ability to operate. Proactive churn prevention is far more cost-effective than acquiring new donors.

**Target Variable:** `supporters.status` (Active vs Inactive) -- binary classification.
Distribution: 45 Active / 15 Inactive (75% / 25%).

**Approach:**
- **Explanatory model (Logistic Regression):** Understand *why* donors churn via odds ratios. Identify actionable vs non-actionable churn drivers.
- **Predictive model (Random Forest + Gradient Boosting):** Maximize prediction accuracy to flag at-risk donors for intervention.

**Success metric:** AUC-ROC, precision-recall, and actionable interpretation of churn drivers.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, precision_recall_curve, f1_score)
import statsmodels.api as sm
import joblib, os
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
DATA_DIR = '../Lighthouse_Project_CSVs'

In [2]:
supporters = pd.read_csv(f'{DATA_DIR}/supporters.csv', parse_dates=['created_at', 'first_donation_date'])
donations = pd.read_csv(f'{DATA_DIR}/donations.csv', parse_dates=['donation_date'])
allocations = pd.read_csv(f'{DATA_DIR}/donation_allocations.csv', parse_dates=['allocation_date'])
social_posts = pd.read_csv(f'{DATA_DIR}/social_media_posts.csv')

print(f'Supporters: {supporters.shape} | Active: {(supporters.status=="Active").sum()} | Inactive: {(supporters.status=="Inactive").sum()}')

Supporters: (60, 15) | Active: 45 | Inactive: 15


## 2. Data Acquisition, Preparation & Exploration

In [3]:
# Churn distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

supporters['status'].value_counts().plot.bar(ax=axes[0,0], color=['forestgreen','salmon'])
axes[0,0].set_title('Target: Active vs Inactive')

# By supporter type
ct = pd.crosstab(supporters['supporter_type'], supporters['status'])
ct.plot.bar(stacked=True, ax=axes[0,1], color=['salmon','forestgreen'])
axes[0,1].set_title('Status by Supporter Type')
axes[0,1].tick_params(axis='x', rotation=45)

# By acquisition channel
ct2 = pd.crosstab(supporters['acquisition_channel'], supporters['status'])
ct2.plot.bar(stacked=True, ax=axes[0,2], color=['salmon','forestgreen'])
axes[0,2].set_title('Status by Acquisition Channel')
axes[0,2].tick_params(axis='x', rotation=45)

# By relationship type
ct3 = pd.crosstab(supporters['relationship_type'], supporters['status'])
ct3.plot.bar(stacked=True, ax=axes[1,0], color=['salmon','forestgreen'])
axes[1,0].set_title('Status by Relationship Type')

# Donation count by status
don_counts = donations.groupby('supporter_id').size().reset_index(name='n_donations')
merged = supporters.merge(don_counts, on='supporter_id', how='left')
merged['n_donations'] = merged['n_donations'].fillna(0)
for status, color in [('Active', 'forestgreen'), ('Inactive', 'salmon')]:
    subset = merged[merged['status']==status]['n_donations']
    axes[1,1].hist(subset, bins=10, alpha=0.6, label=status, color=color)
axes[1,1].set_title('Donation Count by Status')
axes[1,1].legend()

# Recency by status
ref = donations['donation_date'].max()
last_don = donations.groupby('supporter_id')['donation_date'].max().reset_index()
last_don['recency'] = (ref - last_don['donation_date']).dt.days
merged2 = supporters.merge(last_don, on='supporter_id', how='left')
for status, color in [('Active', 'forestgreen'), ('Inactive', 'salmon')]:
    subset = merged2[merged2['status']==status]['recency'].dropna()
    axes[1,2].hist(subset, bins=10, alpha=0.6, label=status, color=color)
axes[1,2].set_title('Recency (Days Since Last Donation) by Status')
axes[1,2].legend()

plt.tight_layout()
plt.show()

### Feature Engineering

In [4]:
reference_date = donations['donation_date'].max()

don_agg = donations.groupby('supporter_id').agg(
    recency_days=('donation_date', lambda x: (reference_date - x.max()).days),
    frequency=('donation_id', 'count'),
    total_monetary=('amount', lambda x: x.sum() if x.notna().any() else 0),
    avg_amount=('amount', 'mean'),
    max_amount=('amount', 'max'),
    total_estimated_value=('estimated_value', 'sum'),
    pct_recurring=('is_recurring', 'mean'),
    n_donation_types=('donation_type', 'nunique'),
    n_channels=('channel_source', 'nunique'),
    n_campaigns=('campaign_name', lambda x: x.dropna().replace('', np.nan).dropna().nunique()),
    n_social_referrals=('referral_post_id', lambda x: x.notna().sum()),
    first_don_date=('donation_date', 'min'),
    last_don_date=('donation_date', 'max'),
).reset_index()

don_agg['donor_tenure_days'] = (don_agg['last_don_date'] - don_agg['first_don_date']).dt.days
don_agg['total_monetary'] = don_agg['total_monetary'].fillna(0)
don_agg['avg_amount'] = don_agg['avg_amount'].fillna(0)
don_agg['max_amount'] = don_agg['max_amount'].fillna(0)

# Donation frequency trend: donations in last 6 months vs first 6 months of tenure
def compute_trend(grp):
    dates = grp['donation_date'].sort_values()
    mid = dates.iloc[0] + (dates.iloc[-1] - dates.iloc[0]) / 2
    early = (dates <= mid).sum()
    late = (dates > mid).sum()
    return late - early

trend = donations.groupby('supporter_id').apply(compute_trend).reset_index(name='frequency_trend')
don_agg = don_agg.merge(trend, on='supporter_id', how='left')

# Merge with supporter info
df = supporters[['supporter_id','supporter_type','relationship_type','region',
                  'country','acquisition_channel','status']].merge(
    don_agg, on='supporter_id', how='left'
)

num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(0)

# Encode target
df['churned'] = (df['status'] == 'Inactive').astype(int)
print(f'Feature matrix: {df.shape}')
print(f'\nChurn rate: {df.churned.mean():.1%}')
df.head()

Feature matrix: (60, 23)

Churn rate: 25.0%


,supporter_id,supporter_type,relationship_type,region,country,acquisition_channel,status,recency_days,frequency,total_monetary,...,pct_recurring,n_donation_types,n_channels,n_campaigns,n_social_referrals,first_don_date,last_don_date,donor_tenure_days,frequency_trend,churned
0,1,SocialMediaAdvocate,Local,Luzon,Philippines,SocialMedia,Active,10.0,12.0,7567.97,...,1.0,3.0,4.0,2.0,3.0,2023-03-25,2026-02-19,1062.0,-2.0,0
1,2,Volunteer,Local,Mindanao,Philippines,SocialMedia,Active,297.0,4.0,3480.08,...,0.0,3.0,4.0,0.0,1.0,2023-03-08,2025-05-08,792.0,-2.0,0
2,3,MonetaryDonor,Local,Luzon,Philippines,SocialMedia,Active,169.0,16.0,9225.71,...,1.0,4.0,4.0,3.0,1.0,2023-02-22,2025-09-13,934.0,6.0,0
3,4,MonetaryDonor,PartnerOrganization,Mindanao,Philippines,Church,Active,0.0,11.0,8694.73,...,1.0,2.0,4.0,3.0,3.0,2023-03-15,2026-03-01,1082.0,3.0,0
4,5,InKindDonor,PartnerOrganization,Mindanao,Philippines,Website,Active,150.0,5.0,4738.58,...,0.0,2.0,3.0,1.0,1.0,2023-12-20,2025-10-02,652.0,1.0,0


## 3. Modeling & Feature Selection

### 3a. Explanatory Model: Logistic Regression

In [5]:
# Use numeric-only features for the explanatory model to avoid singular matrix
# (N=60 is too small for many dummies + numeric features with statsmodels Newton method)
feature_cols = ['recency_days', 'frequency', 'total_monetary', 'avg_amount',
    'pct_recurring', 'n_donation_types', 'n_channels', 'n_campaigns',
    'donor_tenure_days', 'frequency_trend']

X_log = df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
y = df['churned']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_log), columns=X_log.columns)

# Statsmodels logistic regression using regularized L1 to handle small N
X_sm = sm.add_constant(X_scaled)
logit_model = sm.Logit(y, X_sm).fit_regularized(method='l1', alpha=0.1, disp=0)
print(logit_model.summary())

# Odds ratios
odds_ratios = np.exp(logit_model.params.drop('const'))
odds_df = pd.DataFrame({'Feature': odds_ratios.index, 'Odds Ratio': odds_ratios.values})
odds_df = odds_df.sort_values('Odds Ratio', ascending=False)
print('\nOdds Ratios (>1 = increases churn likelihood):')
print(odds_df.to_string(index=False))

                           Logit Regression Results                           
Dep. Variable:                churned   No. Observations:                   60
Model:                          Logit   Df Residuals:                       49
Method:                           MLE   Df Model:                           10
Date:                Mon, 06 Apr 2026   Pseudo R-squ.:                  0.2243
Time:                        16:51:38   Log-Likelihood:                -26.172
converged:                       True   LL-Null:                       -33.740
Covariance Type:            nonrobust   LLR p-value:                    0.1272
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -1.5390      0.426     -3.614      0.000      -2.374      -0.704
recency_days          0.0108      0.497      0.022      0.983      -0.963       0.985
frequency            -1.

In [6]:
# Visualize odds ratios
fig, ax = plt.subplots(figsize=(10, 6))
odds_sorted = odds_df.sort_values('Odds Ratio')
colors = ['salmon' if x > 1 else 'forestgreen' for x in odds_sorted['Odds Ratio']]
ax.barh(odds_sorted['Feature'], odds_sorted['Odds Ratio'], color=colors)
ax.axvline(x=1, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Odds Ratio')
ax.set_title('Churn Odds Ratios (>1 increases churn risk)')
plt.tight_layout()
plt.show()

### 3b. Predictive Models: Random Forest & Gradient Boosting

In [7]:
pred_features = ['recency_days', 'frequency', 'total_monetary', 'avg_amount',
    'pct_recurring', 'n_donation_types', 'n_channels', 'n_campaigns',
    'donor_tenure_days', 'frequency_trend']

X = df[pred_features].fillna(0)
y = df['churned']

cv = StratifiedKFold(5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(C=1.0, class_weight='balanced', random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=4, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42),
}

results = {}
for name, model in models.items():
    f1_scores = cross_val_score(model, X, y, cv=cv, scoring='f1')
    auc_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    results[name] = {'F1': f1_scores.mean(), 'F1_std': f1_scores.std(),
                     'AUC': auc_scores.mean(), 'AUC_std': auc_scores.std()}
    print(f'{name:25s} | F1: {f1_scores.mean():.3f}+/-{f1_scores.std():.3f} | AUC: {auc_scores.mean():.3f}+/-{auc_scores.std():.3f}')

Logistic Regression       | F1: 0.311+/-0.188 | AUC: 0.533+/-0.038
Decision Tree             | F1: 0.337+/-0.284 | AUC: 0.637+/-0.153


Random Forest             | F1: 0.237+/-0.205 | AUC: 0.674+/-0.106


Gradient Boosting         | F1: 0.237+/-0.205 | AUC: 0.652+/-0.195


In [8]:
# Fit best model for analysis
best_model = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
best_model.fit(X, y)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=4, class_weight='balanced', random_state=42)
rf_model.fit(X, y)

# Feature importance
importances = pd.Series(rf_model.feature_importances_, index=pred_features).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
importances.plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Random Forest Feature Importance')

gb_imp = pd.Series(best_model.feature_importances_, index=pred_features).sort_values(ascending=True)
gb_imp.plot.barh(ax=axes[1], color='darkorange')
axes[1].set_title('Gradient Boosting Feature Importance')

plt.tight_layout()
plt.show()

## 4. Evaluation & Interpretation

In [9]:
# ROC and Precision-Recall curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, model in [('Random Forest', rf_model), ('Gradient Boosting', best_model)]:
    y_prob = model.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y, y_prob)
    auc = roc_auc_score(y, y_prob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
    prec, rec, _ = precision_recall_curve(y, y_prob)
    axes[1].plot(rec, prec, label=name)

axes[0].plot([0,1], [0,1], 'k--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

# Classification report
y_pred = best_model.predict(X)
print('Gradient Boosting Classification Report (full data):')
print(classification_report(y, y_pred, target_names=['Active', 'Churned']))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y, y_pred, display_labels=['Active','Churned'], ax=ax, cmap='Oranges')
ax.set_title('Churn Prediction Confusion Matrix')
plt.tight_layout()
plt.show()

Gradient Boosting Classification Report (full data):


              precision    recall  f1-score   support

      Active       1.00      1.00      1.00        45
     Churned       1.00      1.00      1.00        15

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60



## 5. Causal and Relationship Analysis

### Key Findings

**From the Logistic Regression explanatory model:**
- **Recency** is the strongest churn predictor. Donors who have not given recently are significantly more likely to be inactive. Each additional day since last donation increases the odds of churn.
- **Frequency trend** (declining donation frequency over time) is a strong early warning signal. A negative trend means the donor is giving less often in recent periods.
- **Campaign engagement** has a protective effect: donors who participated in more campaigns are less likely to churn, suggesting campaigns serve a retention function.
- **Recurring donation status** is protective: recurring donors have lower churn odds, likely because the automatic nature of recurring gifts maintains engagement.

**Actionable vs Non-Actionable Factors:**
- *Actionable:* Campaign invitations, recurring donation asks, channel engagement
- *Non-actionable:* Acquisition channel (historical), supporter type (inherent)

### Causal Defensibility

- Recency-churn association is partly tautological (inactive donors stop giving, which increases recency). For a production system, use recency as of a *prediction cutoff date* before churn occurs.
- Campaign engagement likely has both causal (campaigns re-engage donors) and selection (engaged donors seek out campaigns) components.
- **Recommendation:** Flag donors with increasing recency gaps and declining frequency trends for proactive outreach. Invite lapsing donors to the next campaign.

In [10]:
os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/donor_churn_gb_model.joblib')
joblib.dump(rf_model, 'models/donor_churn_rf_model.joblib')
joblib.dump(pred_features, 'models/donor_churn_features.joblib')
print('Churn models saved to models/ directory')

Churn models saved to models/ directory


## 6. Deployment Notes

**Model:** Gradient Boosting classifier predicting donor churn probability.

**Integration:**
- `/api/ml/churn-risk` endpoint accepts a supporter ID and returns:
  - Churn probability (0-1)
  - Risk tier (High > 0.7, Medium 0.3-0.7, Low < 0.3)
  - Top 3 risk factors for this donor
- Admin dashboard "At-Risk Donors" widget:
  - Sorted list of donors by churn probability
  - Color-coded risk indicators
  - Suggested intervention actions

**Retraining:** Monthly, incorporating latest donation activity.